# Evaluate Testing Matrix With TTA

Notebook nay chay testing cho cac tap:

- `ffpp-test`
- `ffpp-test-corruption`
- `celebdfv1-test-corruption`
- `ffpp-test-balanced`
- `ffpp-test-corruption-balanced`
- `celebdfv1-test-corruption-balanced`

Models:

- `linear-probe`
- `osd`
- `linear-probe-balanced`
- `osd-balanced`

TTA methods: `none` va tat ca method co san trong `deepfake_tta.methods.registry`.


## Kaggle Setup

Clone repo va install package local.

In [ ]:
!git clone -b dev https://github.com/hoavien0110/training-free-tta-for-deepfake-detection.git /kaggle/working/training-free-tta-for-deepfake-detection
%cd /kaggle/working/training-free-tta-for-deepfake-detection
# !pip install -q -e . --no-deps

## Check Mounted Inputs

Neu Kaggle slug khac, sua path trong cell `Run Testing Matrix`.

In [ ]:
# !find /kaggle/input -maxdepth 4 -type f \
#   \( -name "*.pt" -o -name "*.csv" \) | sort | sed -n "1,220p"

## Run Testing Matrix

`ffpp-test-balanced` duoc tao bang cach undersample `ffpp_test_features.pt` theo REAL/FAKE. Hai corruption balanced datasets dung aligned sample IDs chung cho moi corruption file.

In [ ]:
FFPP_SPLIT = "/kaggle/input/ffpp-split-features"
FFPP_CORR = "/kaggle/input/ffpp-test-embeddings/ffpp_test_embeddings"
CELEB_CORR = "/kaggle/input/deepfakebench-features"
MODELS = "/kaggle/input/ffpp-training-free-models"
MODELS_BAL = "/kaggle/input/ffpp-training-free-models-balanced"

TTA_METHODS = " ".join([
    "none",
    "tip_adapter",
    # "boost_adapter",
    "freetta",
    "freetta_linear_ensemble",
    # "compact_cache_adapter",
    # "online_confident_cache_adapter",
    # "crg",
    # "dmn",
    # "dpe",
    # "dota",
    # "freetta_balanced",
    # "freetta_linear_ensemble_balanced",
    # "bca_balanced",
    # "bca_linear_ensemble_balanced",
    # "dpe_balanced",
    # "dpe_linear_ensemble_balanced",
    # "dota_balanced",
    # "dota_linear_ensemble_balanced",
    # "prototype_linear_tta_balanced",
    # "online_cache_10_balanced",
    "bca",
    # "dynaprompt",
    # "prototype_linear_tta",
])

!python testing/evaluate_tta_matrix.py \
  --train-features {FFPP_SPLIT}/ffpp_train_features.pt \
  --dataset ffpp-test={FFPP_SPLIT}/ffpp_test_features.pt \
  --dataset ffpp-test-corruption={FFPP_CORR} \
  --dataset celebdfv1-test-corruption={CELEB_CORR} \
  --dataset ffpp-test-balanced={FFPP_SPLIT}/ffpp_test_features.pt \
  --dataset ffpp-test-corruption-balanced={FFPP_CORR} \
  --dataset celebdfv1-test-corruption-balanced={CELEB_CORR} \
  --balanced-dataset ffpp-test-balanced \
  --balanced-aligned-dataset ffpp-test-corruption-balanced \
  --balanced-aligned-dataset celebdfv1-test-corruption-balanced \
  --model linear-probe={MODELS}/ffpp_linear_probe_split.pt \
  --model osd={MODELS}/ffpp_osd_linear_probe_split.pt \
  --model linear-probe-balanced={MODELS_BAL}/ffpp_linear_probe_split.pt \
  --model osd-balanced={MODELS_BAL}/ffpp_osd_linear_probe_split.pt \
  --thresholds-csv linear-probe={MODELS}/ffpp_probe_thresholds.csv \
  --thresholds-csv osd={MODELS}/ffpp_probe_thresholds.csv \
  --thresholds-csv linear-probe-balanced={MODELS_BAL}/ffpp_probe_thresholds.csv \
  --thresholds-csv osd-balanced={MODELS_BAL}/ffpp_probe_thresholds.csv \
  --tta-methods {TTA_METHODS} \
  --method-cache-dir /kaggle/working/tta_method_cache \
  --results-output /kaggle/working/tta_testing_matrix_results.csv \
  --eval-batch-size 4096 \
  --test-batch-size 512 \
  --cache-batch-size 8192 \
  --tip-adapter-shots-per-class 16 \
  --block-size 10 \
  --shuffle-all-tests \
  --device auto \
  --continue-on-error

## Preview Results

In [ ]:
import pandas as pd

results = pd.read_csv("/kaggle/working/tta_testing_matrix_results.csv")
display(results.head())
display(results.sort_values(["dataset", "level", "corruption", "model", "method"]))

## Export Compact Tables

In [ ]:
metric_cols = ["acc", "f1", "auc", "ap", "eer"]
summary = results.groupby(["dataset", "model", "method"], dropna=False)[metric_cols].mean().reset_index()
summary.to_csv("/kaggle/working/tta_testing_matrix_summary.csv", index=False)
display(summary.sort_values(["dataset", "model", "f1"], ascending=[True, True, False]))